# scRNA-seq Clustering

This notebook continues from the dimensionality-reduction workflow and identifies groups of transcriptionally similar cells with Leiden clustering.

Input: `data/s4d8_dimensionality_reduction.h5ad`

Workflow:
1. Load the dimensionality-reduced `AnnData` object.
2. Rebuild the neighborhood graph using the top 30 principal components.
3. Recompute UMAP.
4. Run Leiden clustering at multiple resolutions.
5. Compare cluster structures on the same UMAP.
6. Save the clustered dataset to `results/s4d8_clustered.h5ad`.


## Environment setup

In [ ]:
from pathlib import Path
import scanpy as sc

sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)


## Load the dimensionality-reduction results

In [ ]:
DATA_PATH = Path("data/s4d8_dimensionality_reduction.h5ad")
adata = sc.read_h5ad(DATA_PATH)
adata


## Build the neighborhood graph

In [ ]:
# Construct the KNN graph from the first 30 principal components
sc.pp.neighbors(adata, n_pcs=30)


## UMAP embedding

In [ ]:
# Recompute UMAP from the updated neighborhood graph
sc.tl.umap(adata)


## Leiden clustering

In [ ]:
# Default Leiden clustering
sc.tl.leiden(adata, flavor="igraph", n_iterations=2)


## Compare clustering resolutions

In [ ]:
# Coarse clustering
sc.tl.leiden(adata, key_added="leiden_res0_25", resolution=0.25, flavor="igraph", n_iterations=2)


In [ ]:
# Intermediate clustering
sc.tl.leiden(adata, key_added="leiden_res0_5", resolution=0.5, flavor="igraph", n_iterations=2)


In [ ]:
# Finer clustering
sc.tl.leiden(adata, key_added="leiden_res1", resolution=1.0, flavor="igraph", n_iterations=2)


In [ ]:
# Compare resolutions using the same UMAP embedding
sc.pl.umap(adata, color=["leiden_res0_25", "leiden_res0_5", "leiden_res1"], legend_loc="on data")


## Save the clustered dataset

In [ ]:
OUTPUT_PATH = Path("results/s4d8_clustered.h5ad")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
adata.write_h5ad(OUTPUT_PATH)
